# 주11: 백테스트

**202315163 김소희**

삼성전자(005930)를 처음에 매수한 뒤 계속 보유하는 전략과,
12일선과 26일선의 교차에 따라 사고파는 전략을 비교한다.
일봉 데이터로 백테스트하고 누적 수익률, 최대 낙폭(MDD), 샤프 지수를 계산한다.

## 1. 분석 조건

- 기간: **2025-01-02 ~ 2026-11-02**. 11월 첫 거래일까지의 데이터를 사용한다.
- 초기 자본: 두 전략 모두 **10,000,000원**.
- 수수료: 기본값은 **0%** 이다. 0.015%와 1.5%일 때의 결과도 비교한다. 세금과 슬리피지는 반영하지 않는다.
- 체결 가격: 매매 신호가 나온 날의 종가로 거래한다.
- 매수 수량: 주식은 1주 단위로 산다. 보유 현금으로 매수대금과 수수료를 모두 낼 수 있는 만큼 사고, 대출과 공매도는 하지 않는다.
- 종료일 평가: 남아 있는 주식은 팔지 않고 마지막 종가로 평가한다. 미실현손익도 수익률에 포함한다.
- 배당금: 수익에 포함하지 않는다.
- 샤프 지수: 연 **250거래일**, 무위험 수익률 **0%** 로 계산한다.

In [1]:
import FinanceDataReader as fdr
import pandas as pd
import numpy as np
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
from backtesting.test import SMA

C:\Users\wneh2\Desktop\bigdata-ljs\.omx\verification\wk11\env\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

## 2. 삼성전자 일봉 데이터
삼성전자의 일별 시가, 고가, 저가, 종가, 거래량을 조회한다.

In [2]:
start = '2025-01-02'
end = '2026-11-02'
data = fdr.DataReader('005930', start, end)
data = data[['Open', 'High', 'Low', 'Close', 'Volume']]

cash = 10000000
commission = 0.0

print('종목: 삼성전자 (005930)')
print('실제 데이터 시작일:', data.index[0])
print('실제 데이터 종료일:', data.index[-1])
print('거래일 수:', len(data))
data.head()

종목: 삼성전자 (005930)
실제 데이터 시작일: 2025-01-02 00:00:00
실제 데이터 종료일: 2026-11-02 00:00:00
거래일 수: 445


,Open,High,Low,Close,Volume
Date,,,,,
2025-01-02,52700,53600,52300,53400,16630538
2025-01-03,52800,55100,52800,54400,19318046
2025-01-06,54400,56200,54300,55900,19034284
2025-01-07,56800,57300,55400,55400,17030235
2025-01-08,54800,57500,54700,57300,26593553


In [3]:
data.tail()

,Open,High,Low,Close,Volume
Date,,,,,
2026-10-27,222000,225500,220000,225000,14196879
2026-10-28,231000,247000,228000,245000,17840544
2026-10-29,241000,249000,226000,230500,24975596
2026-10-30,223500,229500,216500,220500,19668075
2026-11-02,219000,232000,217000,230000,14537981


## 3. 전략 A: Buy & Hold
2025년 1월 2일 종가로 주식을 한 번 매수하고 종료일까지 보유한다.
초기 자본으로 매수대금과 수수료를 낼 수 있는 만큼 산다.

In [4]:
first_close = data['Close'].iloc[0]
buy_size = int(cash / (first_close * (1 + commission)))

class BuyHold(Strategy):
    def init(self):
        self.buy(size=buy_size)

    def next(self):
        pass

## 4. 전략 B: 이동평균 Crossover
- 주식을 보유하지 않은 상태에서 12일 단순이동평균선이 26일선을 상향 돌파하면 매수한다.
- 주식을 보유한 상태에서 12일선이 26일선을 하향 이탈하면 전량 매도한다.
- 매도 후에는 현금을 보유하고 다음 매수 신호를 기다린다.
- 당일과 전일의 26일 이동평균이 모두 계산되는 날부터 매매 신호를 확인한다. 그전에는 현금을 보유하므로 수익률은 0%다.
- 매매 신호는 종가로 판단하고 같은 날 종가에 체결된다고 가정한다.

In [5]:
class SmaCross(Strategy):
    n1 = 12
    n2 = 26

    def init(self):
        self.ma1 = self.I(SMA, self.data.Close, self.n1)
        self.ma2 = self.I(SMA, self.data.Close, self.n2)

    def next(self):
        if not self.position and crossover(self.ma1, self.ma2):
            self.buy()
        elif self.position and crossover(self.ma2, self.ma1):
            self.position.close()

## 5. 두 전략 백테스트 실행
두 전략은 초기 자본과 수수료를 같게 설정한다.
매매 가격은 당일 종가로 정하고, 종료일에 남은 주식도 마지막 종가로 평가한다.

In [6]:
bt_a = Backtest(data, BuyHold, cash=cash, commission=commission,
                trade_on_close=True, exclusive_orders=True)
stats_a = bt_a.run()

bt_b = Backtest(data, SmaCross, cash=cash, commission=commission,
                trade_on_close=True, exclusive_orders=True)
stats_b = bt_b.run()

print('A 첫날 매수 수량:', buy_size, '주')
print('A 첫날 매수 가격:', first_close, '원')
print('A 최종 평가액:', round(stats_a['Equity Final [$]'], 2), '원')
print('B 최종 평가액:', round(stats_b['Equity Final [$]'], 2), '원')
print('B의 완료 거래 횟수(매수 후 매도):', int(stats_b['# Trades']), '회')

C:\Users\wneh2\AppData\Local\Temp\ipykernel_17536\3869562728.py:3: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats_a = bt_a.run()


A 첫날 매수 수량: 187 주
A 첫날 매수 가격: 53400 원
A 최종 평가액: 43024200.0 원
B 최종 평가액: 24748500.0 원
B의 완료 거래 횟수(매수 후 매도): 9 회


## 6. 성과 지표 계산과 비교표
- **누적 수익률:** `(최종 평가액 / 초기 자본 - 1) × 100`으로 계산한다.
- **최대 낙폭(MDD):** 계좌 평가액의 고점 대비 가장 큰 하락률이다. 음수로 표시하며, 절댓값이 클수록 하락폭이 크다.
- **샤프 지수:** `(연환산 평균 수익률 - 무위험 수익률) / 연환산 표준편차`로 계산한다.

계좌 평가액은 `남은 현금 + 보유 수량 × 당일 종가`이다.
A는 초기 자본에 보유 주식의 평가손익을 더하고 매수 수수료를 뺀다.
B는 백테스트에서 계산한 계좌 평가액을 사용한다.

일별 수익률은 `당일 평가액 / 전일 평가액 - 1`이다.
첫날은 전일 평가액 대신 초기 자본을 넣는다.

In [7]:
# A: 종가 변화의 누적합으로 보유 수익 계산
earnDiffDaily = data['Close'].diff()
earnDiffBuyHold = earnDiffDaily.fillna(0).cumsum()
equity_a = cash + buy_size * earnDiffBuyHold - buy_size * first_close * commission

# B: 백테스트 계좌 평가액
equity_b = stats_b['_equity_curve']['Equity']

# 두 전략에 같은 일별 수익률 계산식 적용
returns_a = equity_a / equity_a.shift(1).fillna(cash) - 1
returns_b = equity_b / equity_b.shift(1).fillna(cash) - 1

# 샤프 지수: 연평균 초과수익 / 연간 표준편차
tradingDays = 250
riskFreeRate = 0.0

returnYearly_a = np.mean(returns_a) * tradingDays
stdYearly_a = np.std(returns_a) * np.sqrt(tradingDays)
sharpe_a = (returnYearly_a - riskFreeRate) / stdYearly_a

returnYearly_b = np.mean(returns_b) * tradingDays
stdYearly_b = np.std(returns_b) * np.sqrt(tradingDays)
sharpe_b = (returnYearly_b - riskFreeRate) / stdYearly_b

result = pd.DataFrame({
    '전략': ['A: Buy & Hold', 'B: SMA(12, 26) Crossover'],
    '누적 수익률 (%)': [(stats_a['Equity Final [$]'] / cash - 1) * 100,
                   (stats_b['Equity Final [$]'] / cash - 1) * 100],
    'MDD (%)': [stats_a['Max. Drawdown [%]'], stats_b['Max. Drawdown [%]']],
    'Sharpe Ratio': [sharpe_a, sharpe_b]
})
np.round(result, 4)

,전략,누적 수익률 (%),MDD (%),Sharpe Ratio
0,A: Buy & Hold,330.242,-42.8876,1.6317
1,"B: SMA(12, 26) Crossover",147.485,-39.1600,1.3050


### 샤프 지수에 사용한 값
연환산 평균 수익률은 일별 평균 수익률에 250을 곱한 값이다. 복리로 계산한 CAGR과는 다르다.
연환산 표준편차는 일별 수익률의 표준편차에 `√250`을 곱한 값이다.
무위험 수익률이 0%이므로, 아래 표의 수익률을 표준편차로 나누면 샤프 지수가 된다.

In [8]:
sharpe_detail = pd.DataFrame({
    '전략': ['A: Buy & Hold', 'B: SMA(12, 26) Crossover'],
    '연환산 평균 수익률 (%)': [returnYearly_a * 100, returnYearly_b * 100],
    '연환산 표준편차 (%)': [stdYearly_a * 100, stdYearly_b * 100]
})
np.round(sharpe_detail, 4)

,전략,연환산 평균 수익률 (%),연환산 표준편차 (%)
0,A: Buy & Hold,101.0753,61.9434
1,"B: SMA(12, 26) Crossover",62.3747,47.7953


## 7. 수수료에 따른 결과 비교
기간, 매매 규칙, 초기 자본은 그대로 두고 수수료만 바꾼다.
매수와 매도 때 각각 표에 적힌 수수료를 낸다.
0.015%는 `commission=0.00015`, 1.5%는 `commission=0.015`이다.
기본 결과는 수수료 0%로 계산한다. 1.5%는 수수료가 높을 때 결과가 얼마나 달라지는지 비교하기 위해 넣었다.

In [9]:
fee_rows = [
    [0.0, 'A: Buy & Hold', result['누적 수익률 (%)'].iloc[0],
     result['MDD (%)'].iloc[0], sharpe_a],
    [0.0, 'B: SMA(12, 26) Crossover', result['누적 수익률 (%)'].iloc[1],
     result['MDD (%)'].iloc[1], sharpe_b]
]
base_buy_size = buy_size
high_commission = 0.015

for fee in [0.00015, high_commission]:
    # 수수료를 낼 수 있도록 각 조건의 매수 수량을 다시 계산한다.
    buy_size = int(cash / (first_close * (1 + fee)))
    fee_stats_a = Backtest(data, BuyHold, cash=cash, commission=fee,
                          trade_on_close=True, exclusive_orders=True).run()
    fee_stats_b = Backtest(data, SmaCross, cash=cash, commission=fee,
                          trade_on_close=True, exclusive_orders=True).run()

    fee_equity_a = cash + buy_size * earnDiffBuyHold - buy_size * first_close * fee
    fee_equity_b = fee_stats_b['_equity_curve']['Equity']
    if fee == high_commission:
        high_fee_equity_a = fee_equity_a
        high_fee_equity_b = fee_equity_b

    fee_returns_a = fee_equity_a / fee_equity_a.shift(1).fillna(cash) - 1
    fee_returns_b = fee_equity_b / fee_equity_b.shift(1).fillna(cash) - 1
    fee_sharpe_a = (np.mean(fee_returns_a) * tradingDays - riskFreeRate) / (np.std(fee_returns_a) * np.sqrt(tradingDays))
    fee_sharpe_b = (np.mean(fee_returns_b) * tradingDays - riskFreeRate) / (np.std(fee_returns_b) * np.sqrt(tradingDays))

    fee_rows.append([fee * 100, 'A: Buy & Hold',
                     (fee_stats_a['Equity Final [$]'] / cash - 1) * 100,
                     fee_stats_a['Max. Drawdown [%]'], fee_sharpe_a])
    fee_rows.append([fee * 100, 'B: SMA(12, 26) Crossover',
                     (fee_stats_b['Equity Final [$]'] / cash - 1) * 100,
                     fee_stats_b['Max. Drawdown [%]'], fee_sharpe_b])

buy_size = base_buy_size
fee_comparison = pd.DataFrame(fee_rows, columns=[
    '편도 수수료 (%)', '전략', '누적 수익률 (%)', 'MDD (%)', 'Sharpe Ratio'
])
np.round(fee_comparison, 4)

C:\Users\wneh2\AppData\Local\Temp\ipykernel_17536\1287109426.py:14: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  trade_on_close=True, exclusive_orders=True).run()


C:\Users\wneh2\AppData\Local\Temp\ipykernel_17536\1287109426.py:14: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  trade_on_close=True, exclusive_orders=True).run()


,편도 수수료 (%),전략,누적 수익률 (%),MDD (%),Sharpe Ratio
0,0.000,A: Buy & Hold,330.2420,-42.8876,1.6317
1,0.000,"B: SMA(12, 26) Crossover",147.4850,-39.1600,1.3050
2,0.015,A: Buy & Hold,330.2270,-42.8885,1.6317
3,0.015,"B: SMA(12, 26) Crossover",145.2355,-39.3163,1.2960
4,1.500,A: Buy & Hold,323.4702,-42.8792,1.6175
5,1.500,"B: SMA(12, 26) Crossover",88.4921,-43.6752,0.9842


## 8. 결과 해석
수수료가 **0%** 일 때 누적 수익률은 A가 **330.24%**, B가 **147.49%** 로, A가 B의 약 **2.24배**다.  
A는 상승장 내내 주식을 보유했다. B는 이동평균의 매수 신호를 기다렸고, 재매수 8번 중 7번을 직전 매도가보다 비싸게 샀다. 초반 세 거래에서도 각각 -3.51%, -8.76%, -1.23%의 손실을 봤다.  
샤프 지수는 A가 **1.63**, B가 **1.31**로, 수익률만큼 차이가 크지는 않다. 최대 낙폭과 수익률의 변동성은 B가 더 작았다.  
수수료를 **1.5%** 로 높이면 샤프 지수는 A가 **1.62**, B가 **0.98**로 차이가 더 커진다.

### 거래 내역
아래 표는 수수료 0%일 때 B의 매수일, 매도일, 매수가, 매도가, 거래 수익률이다.
`ReturnPct`는 거래 수익률이며, -0.25는 -25%를 뜻한다.

In [10]:
trade_detail = stats_b['_trades'][[
    'EntryTime', 'ExitTime', 'EntryPrice', 'ExitPrice', 'ReturnPct'
]].copy()
trade_detail['ReturnPct'] = np.round(trade_detail['ReturnPct'], 4)
trade_detail

,EntryTime,ExitTime,EntryPrice,ExitPrice,ReturnPct
0,2025-02-18,2025-03-12,56900.0,54900,-0.0351
1,2025-03-24,2025-04-11,60500.0,55200,-0.0876
2,2025-05-13,2025-05-30,56900.0,56200,-0.0123
3,2025-06-09,2025-09-01,59800.0,67600,0.1304
4,2025-09-15,2025-11-20,76500.0,100600,0.3150
5,2025-12-05,2026-03-24,108400.0,189700,0.7500
6,2026-04-13,2026-07-06,201000.0,318000,0.5821
7,2026-08-20,2026-09-15,271000.0,250500,-0.0756
8,2026-09-22,2026-10-15,277500.0,207500,-0.2523


### 현금 보유 기간의 주가
B는 2026년 7월 6일 318,000원에 매도한 뒤 8월 20일 271,000원에 다시 매수했다.
그 사이 7월 30일 종가는 207,000원까지 떨어졌다. B는 이때 주식을 보유하지 않아 추가 손실을 보지 않았다.
아래 표에서 이 날짜들의 종가와 마지막 거래의 매수가, 매도가를 확인할 수 있다.

In [11]:
price_detail = data.loc[
    ['2026-07-06', '2026-07-30', '2026-08-20', '2026-09-22', '2026-10-15'],
    ['Close']
]
price_detail

,Close
Date,
2026-07-06,318000
2026-07-30,207000
2026-08-20,271000
2026-09-22,277500
2026-10-15,207500


### MDD의 고점과 저점
MDD는 계좌 평가액이 이전 고점에서 가장 많이 떨어졌을 때의 하락률이다. 한 거래의 손실률과는 다르다.
매일 그날까지의 최고 평가액과 당일 평가액을 비교해 하락폭이 가장 큰 구간을 찾는다.
아래 표의 MDD는 `(저점 평가액 / 고점 평가액 - 1) × 100`이다.

In [12]:
mdd_rows = []

for fee_pct, strategy, equity in [
    (0.0, 'A: Buy & Hold', equity_a),
    (0.0, 'B: SMA(12, 26) Crossover', equity_b),
    (high_commission * 100, 'A: Buy & Hold', high_fee_equity_a),
    (high_commission * 100, 'B: SMA(12, 26) Crossover', high_fee_equity_b)
]:
    peak_value = cash
    peak_date = equity.index[0]
    mdd = 0.0
    mdd_peak_date = peak_date
    mdd_trough_date = peak_date
    mdd_peak_value = cash
    mdd_trough_value = cash

    for i in range(len(equity)):
        value = equity.iloc[i]
        if value > peak_value:
            peak_value = value
            peak_date = equity.index[i]

        drawdown = (value / peak_value - 1) * 100
        if drawdown < mdd:
            mdd = drawdown
            mdd_peak_date = peak_date
            mdd_trough_date = equity.index[i]
            mdd_peak_value = peak_value
            mdd_trough_value = value

    mdd_rows.append([
        fee_pct, strategy, str(mdd_peak_date)[:10], str(mdd_trough_date)[:10],
        mdd_peak_value, mdd_trough_value, mdd
    ])

mdd_detail = pd.DataFrame(mdd_rows, columns=[
    '편도 수수료 (%)', '전략', '고점 날짜', '저점 날짜',
    '고점 평가액 (원)', '저점 평가액 (원)', 'MDD (%)'
])
np.round(mdd_detail, 4)

,편도 수수료 (%),전략,고점 날짜,저점 날짜,고점 평가액 (원),저점 평가액 (원),MDD (%)
0,0.0,A: Buy & Hold,2026-06-18,2026-07-30,67801700.0,38723200.0,-42.8876
1,0.0,"B: SMA(12, 26) Crossover",2026-06-18,2026-10-15,40678000.0,24748500.0,-39.1600
2,1.5,A: Buy & Hold,2026-06-18,2026-07-30,66727016.0,38115016.0,-42.8792
3,1.5,"B: SMA(12, 26) Crossover",2026-06-18,2026-10-16,33465213.0,18849210.5,-43.6752


수수료 0%에서 B의 MDD **-39.16%** 는 2026년 6월 18일 고점부터 10월 15일 저점까지의 하락률이다.
7월 6일에 매도하기 전까지 평가액이 떨어졌고, 8월 20일과 9월 22일에 다시 매수한 두 거래에서도 손실을 봤다.
마지막 거래에서는 9월 22일 277,500원에 매수해 10월 15일 207,500원에 매도하면서 **-25.23%** 의 손실을 봤다.
MDD에는 이 거래의 손실뿐 아니라 그전에 발생한 평가액 하락도 포함된다.

수수료가 1.5%이면 B의 MDD는 **-43.68%** 이고, 저점은 10월 16일이다.
백테스트에서는 10월 15일 매도의 수수료가 10월 16일 평가액에 반영된다.
A는 한 번만 매수했고, B는 매수와 매도를 각각 9번 했다.
B는 수수료를 더 자주 내므로, 수수료를 높이면 수익률과 샤프 지수가 A보다 더 크게 낮아진다.